In [ ]:
%load_ext autoreload

from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv("../.env")

In [ ]:
%autoreload 2

from pathlib import Path

import geopandas as gpd
import matplotlib.cm as cm
import matplotlib.colors as colors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import rasterio
import seaborn as sns
from scipy.signal import butter, filtfilt

from estuary.util.img import (
    false_color,
)

In [ ]:
gdf = gpd.read_file("/Volumes/x10pro/estuary/geos/ca_data_w_empa_pmep_usgs.geojson")
gdf = gdf[~gdf.skipped].copy()
gdf = gdf.set_index("Site code")

# timeseries_path = "/Users/kyledorman/data/results/estuary/train/20251114-130229/merged_all_regions_timeseries_preds.csv"
timeseries_path = (
    "/Users/kyledorman/data/results/estuary/train/20251114-130229/merged_time_series_preds.csv"
)
preds_all = pd.read_csv(Path(timeseries_path))
# normalize times
preds_all["acquired"] = pd.to_datetime(preds_all["acquired"], errors="coerce", utc=True)
preds_all = preds_all.sort_values("acquired").dropna(subset=["acquired"])
preds_all["y_true_prob"] = 0.05
preds_all.loc[preds_all.y_true == 1, "y_true_prob"] = 0.95

matching_sites = {int(i): str(row["siteid"]) for i, row in gdf[gdf.siteid.notna()].iterrows()}

In [ ]:
empa = pl.read_csv("/Volumes/x10pro/estuary/water_data/raw/empa/logger-raw-publish.csv")

# Normalize unit strings (strip whitespace) so comparisons match exactly
empa = empa.with_columns(
    [
        pl.col("raw_conductivity_unit")
        .cast(pl.Utf8)
        .str.strip_chars()
        .alias("raw_conductivity_unit"),
        pl.col("raw_depth_unit").cast(pl.Utf8).str.strip_chars().alias("raw_depth_unit"),
    ]
)

# Convert μS/cm → mS/cm and overwrite both columns (cast numeric first)
empa = empa.with_columns(
    [
        pl.when(pl.col("raw_conductivity_unit") == "uS/cm")
        .then(pl.col("raw_conductivity").cast(pl.Float64, strict=False) / 1000.0)
        .when(pl.col("raw_conductivity_unit") == "mS/cm")
        .then(pl.col("raw_conductivity").cast(pl.Float64, strict=False))
        .otherwise(pl.col("raw_conductivity").cast(pl.Float64, strict=False))
        .alias("raw_conductivity"),
        # Only standardize the unit to mS/cm when the original was a known conductivity unit
        pl.when(pl.col("raw_conductivity_unit").is_in(["uS/cm", "mS/cm"]))
        .then(pl.lit("mS/cm"))
        .otherwise(pl.col("raw_conductivity_unit"))
        .alias("raw_conductivity_unit"),
    ]
)

# Convert cm → m and overwrite both columns (cast numeric first)
empa = empa.with_columns(
    [
        pl.when(pl.col("raw_depth_unit") == "cm")
        .then(pl.col("raw_depth").cast(pl.Float64, strict=False) / 100.0)
        .when(pl.col("raw_depth_unit") == "m")
        .then(pl.col("raw_depth").cast(pl.Float64, strict=False))
        .otherwise(pl.col("raw_depth").cast(pl.Float64, strict=False))
        .alias("raw_depth"),
        pl.when(pl.col("raw_depth_unit").is_in(["cm", "m"]))
        .then(pl.lit("m"))
        .otherwise(pl.col("raw_depth_unit"))
        .alias("raw_depth_unit"),
    ]
)

# define time parsing (try multiple formats)
parsed_dt = pl.coalesce(
    [
        pl.col("samplecollectiontimestamp").str.strptime(
            pl.Datetime, "%d/%m/%Y %H:%M:%S", strict=False
        ),
        pl.col("samplecollectiontimestamp").str.strptime(
            pl.Datetime, "%d/%m/%Y %H:%M:%S%.f", strict=False
        ),
    ]
)

# offsets relative to UTC (Polars doesn’t know “PST/PDT” by name)
# PST = UTC−8, PDT = UTC−7
empa = empa.with_columns([parsed_dt.alias("samplecollectiontimestamp_parsed")])

# apply offset based on timezone
empa = empa.with_columns(
    [
        pl.when(pl.col("samplecollectiontimezone") == "PST")
        .then(pl.col("samplecollectiontimestamp_parsed") + pl.duration(hours=8))  # PST -> UTC
        .when(pl.col("samplecollectiontimezone") == "PDT")
        .then(pl.col("samplecollectiontimestamp_parsed") + pl.duration(hours=7))  # PDT -> UTC
        .when(pl.col("samplecollectiontimezone") == "UTC")
        .then(pl.col("samplecollectiontimestamp_parsed"))
        .otherwise(pl.col("samplecollectiontimestamp_parsed"))
        .alias("samplecollectiontimestamp_utc2")
    ]
)
empa = empa.filter(pl.col("siteid").is_in(list(matching_sites.values())))

In [ ]:
empa_ranges = (
    empa.group_by(["siteid", "sensorid"])
    .agg(
        [
            pl.col("samplecollectiontimestamp_utc2").min().alias("start"),
            pl.col("samplecollectiontimestamp_utc2").max().alias("end"),
        ]
    )
    # Filter groups where (end - start) > 90 days
    .filter((pl.col("end") - pl.col("start")) > pl.duration(days=90))
    .sort(["siteid", "sensorid"])
).to_pandas()

In [ ]:
empa_ranges.siteid.unique()

In [ ]:
SITE = "NC-NAV"
REGION = next((k for k, v in matching_sites.items() if v == SITE))

site_ranges = empa_ranges[empa_ranges["siteid"] == SITE].copy()
site_ranges["duration"] = site_ranges.end - site_ranges.start

site_ranges

In [ ]:
gdf.loc[int(REGION)]

In [ ]:
def keep_longest_contiguous(
    df: pd.DataFrame, time_col: str = "acquired", gap_hours: int = 4, days: int = 7
) -> pd.DataFrame:
    """
    Find all contiguous segments in a time series (gaps > gap_hours separate segments).
    Return only the longest contiguous segment (largest time span).
    """
    out = df.copy()
    out[time_col] = pd.to_datetime(out[time_col], errors="coerce")
    out = out.dropna(subset=[time_col]).sort_values(time_col)
    if out.empty:
        return out.iloc[0:0]
    gap = pd.Timedelta(hours=gap_hours)
    seg_id = (out[time_col].diff() > gap).cumsum()
    print(np.unique(seg_id))
    # Find all segments, keep the one with the largest span
    best_idx = None
    best_span = pd.Timedelta(0)
    for sid in seg_id.unique():
        seg = out[seg_id == sid]
        if seg.empty:
            continue
        span = seg[time_col].max() - seg[time_col].min()
        if span > best_span:
            best_span = span
            best_idx = sid
    if best_idx is None:
        return out.iloc[0:0]
    out = out[seg_id == best_idx]

    # 3) drop the LAST N calendar days (data “gets wonky at the end”)
    last_ts = out[time_col].max()
    stop = last_ts - pd.Timedelta(days=days)
    out = out[out[time_col] < stop]
    first_ts = out[time_col].min()
    start = first_ts + pd.Timedelta(days=days)
    out = out[out[time_col] > start]

    return out.copy()

In [ ]:
site_sensor_data = empa.filter(pl.col("siteid") == SITE).filter(pl.col("sensorid") == "791371")

depth_df_orig = (
    site_sensor_data[["samplecollectiontimestamp_utc2", "raw_depth"]]
    .to_pandas()
    .rename(columns={"samplecollectiontimestamp_utc2": "acquired"})
    .sort_values(["acquired"])
)

depth_df_orig = depth_df_orig.copy()

# Ensure acquired is datetime and sorted
depth_df_orig["acquired"] = pd.to_datetime(depth_df_orig["acquired"], utc=True)
depth_df_orig = depth_df_orig.sort_values("acquired").drop_duplicates("acquired")

depth_df_orig = keep_longest_contiguous(depth_df_orig, days=7).copy()

# Set datetime index
depth_df_orig = depth_df_orig.set_index("acquired")

depth_df_orig.head(5)

In [ ]:
predictions = preds_all[preds_all.region == int(REGION)]
# keep preds within depth time span
predictions = predictions[
    (predictions["acquired"] >= depth_df_orig.index.min())
    & (predictions["acquired"] <= depth_df_orig.index.max())
].copy()
predictions = predictions[predictions.y_pred_unsure == 0]

In [ ]:
img_labels = pd.read_csv("/Users/kyledorman/data/estuary/dataset/time_series.csv")
img_labels = img_labels[img_labels.label != "unsure"].copy()
img_labels["label_idx"] = img_labels.label.map(lambda a: int(a != "closed"))
img_labels["acquired"] = pd.to_datetime(img_labels["acquired"], utc=True)
img_labels = img_labels.sort_values(["region", "acquired"])
img_labels = img_labels[img_labels.region == int(REGION)]
img_labels = img_labels[
    (img_labels["acquired"] >= depth_df_orig.index.min())
    & (img_labels["acquired"] <= depth_df_orig.index.max())
].copy()
img_labels.head()

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 5))

# --- Second y-axis ---
ax1.plot(depth_df_orig.index, depth_df_orig["raw_depth"], color="C1", label="Raw depth", alpha=0.8)
ax1.set_ylabel("raw_depth", color="C1")
ax1.tick_params(axis="y", labelcolor="C1")

# --- First y-axis: predictions ---
ax2 = ax1.twinx()
ax2.scatter(
    img_labels["acquired"], img_labels["label_idx"] + 0.05, color="r", label="Labels", alpha=0.6
)
ax2.scatter(
    predictions["acquired"], predictions["y_prob"], color="C0", label="Predictions", alpha=0.6
)
ax2.set_xlabel("Acquired")
ax2.set_ylabel("Predictions/Labels", color="C0")
ax2.tick_params(axis="y", labelcolor="C0")

plt.title("Labels vs Raw Depth Over Time")
fig.tight_layout()
plt.show()

In [ ]:
ft = depth_df_orig[(depth_df_orig.index > start) & (depth_df_orig.index < end)].reset_index()

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 5))

start = pd.Timestamp(year=2022, month=2, day=1, tz="UTC")
end = pd.Timestamp(year=2022, month=3, day=1, tz="UTC")

ft = depth_df_orig[(depth_df_orig.index > start) & (depth_df_orig.index < end)].reset_index()
pt = predictions[(predictions.acquired > start) & (predictions.acquired < end)]
lt = img_labels[(img_labels.acquired > start) & (img_labels.acquired < end)]

# --- First y-axis: predictions ---
ax1.scatter(pt["acquired"], pt["y_prob"], color="C0", label="Predicted probability", alpha=0.6)
ax1.scatter(lt["acquired"], lt["label_idx"] + 0.05, color="red", label="Label", alpha=0.6)
ax1.set_ylim(-0.1, 1.1)
ax1.set_xlabel("Acquired")
ax1.set_ylabel("y_prob", color="C0")
ax1.tick_params(axis="y", labelcolor="C0")

# --- Second y-axis ---
ax2 = ax1.twinx()
ax2.plot(ft["acquired"], ft["raw_depth"], color="C1", label="Raw depth", alpha=0.8)
ax2.set_ylabel("raw_depth", color="C1")
ax2.tick_params(axis="y", labelcolor="C1")

plt.title("Predictions vs Raw Depth Over Time")
fig.tight_layout()
plt.show()

In [ ]:
row = predictions[(predictions.acquired > start) & (predictions.acquired < end)].iloc[9]
plt.figure(figsize=(7, 7))
with rasterio.open(row.source_tif) as src:
    data = src.read()
    nodata = src.read(1, masked=True).mask
    img = false_color(data, nodata)

plt.axis("off")
plt.imshow(img)
_ = plt.title(f"{row.y_pred} {row.date}")

In [ ]:
depth_df = depth_df_orig.copy()

# Parameters you can tune
rolling_window = "1h"  # local baseline window (1 hour)
drop_threshold = 0.2  # depth drop threshold (in same units as raw_depth)
max_len_points = 20  # max length (in samples) of a "short" invalid drop (e.g. 20 * 6min = 2h)

# 1. Rolling median baseline
baseline = depth_df["raw_depth"].rolling(rolling_window, center=True, min_periods=1).median()

# 2. Residual (how far below/above local baseline)
residual = depth_df["raw_depth"] - baseline

# 3. Boolean mask for drops: residual much LOWER than baseline
drop_mask = residual < -drop_threshold

# 4. Identify contiguous segments of drops
# Each run of True values gets a unique group id
group_id = (drop_mask != drop_mask.shift()).cumsum()

# Count length of each group
group_sizes = drop_mask.groupby(group_id).sum()  # sum works since True=1, False=0

# Find group ids representing short drop segments
short_drop_groups = group_sizes[(group_sizes > 0) & (group_sizes <= max_len_points)].index

# Final mask: only points that are part of short drop segments
short_drop_mask = drop_mask & group_id.isin(short_drop_groups)

print(f"Number of points flagged as short drops: {short_drop_mask.sum()}")

# 5. Set these to NaN
depth_df.loc[short_drop_mask, "raw_depth"] = np.nan

# 6. Interpolate over them (time-based)
depth_df["raw_depth"] = depth_df["raw_depth"].interpolate(method="time")

# Optional: sanity check plot before/after
# import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(depth_df.index, depth_df["raw_depth"], label="Cleaned depth")
ax.scatter(
    depth_df.index[short_drop_mask],
    baseline[short_drop_mask],
    color="red",
    s=50,
    label="Removed drops",
)
ax.legend()
plt.show()

In [ ]:
depth_df = depth_df.copy()

# Build a regular 6-minute grid from start to end
full_index = pd.date_range(
    start=depth_df.index.min(),
    end=depth_df.index.max(),
    freq="6min",  # 6-minute interval
)

# Reindex onto the regular grid
depth_df = depth_df.reindex(full_index)

# Time-based interpolation for raw_depth
depth_df["raw_depth"] = depth_df["raw_depth"].interpolate(method="time")

# Clean up index name
depth_df.index.name = "acquired"

# Optional: quick sanity check
depth_df.head(), depth_df.tail()

In [ ]:
def water_slope(s):
    if len(s) < 5:
        return np.nan
    # slope (linear trend)
    x = s.index.view(int)  # nanoseconds
    y = s.values
    slope = np.polyfit(x, y, 1)[0]

    return slope


def detrended_rms(depth_df, window="12h"):
    s = depth_df["raw_depth"].dropna()

    def detrend_rms(x):
        if len(x) < 5:
            return np.nan
        x = np.array(x)
        t = np.arange(len(x))
        coeff = np.polyfit(t, x, 1)
        detr = x - np.polyval(coeff, t)
        return np.sqrt(np.mean(detr**2))

    return s.rolling(window, center=True).apply(detrend_rms, raw=False)


def bandpass_tidal_rms(depth_df, low_cpd=0.7, high_cpd=3.5, rolling_time="6h"):
    s = depth_df["raw_depth"].dropna()

    # sampling interval
    dt_min = np.median(np.diff(s.index.values).astype("timedelta64[m]").astype(float))
    fs = 60.0 / dt_min  # samples per hour
    fs_per_day = 24 * fs

    # convert cutoff frequencies in cycles/day to Nyquist-normalized frequency
    nyq = 0.5 * fs_per_day
    low = low_cpd / nyq
    high = high_cpd / nyq

    b, a = butter(4, [low, high], btype="band")  # type: ignore
    filtered = filtfilt(b, a, s.values)

    # rolling RMS (3h or 6h)
    rms = (
        pd.Series(filtered, index=s.index)
        .rolling(rolling_time, center=True)
        .apply(lambda x: np.sqrt(np.mean(x**2)))
    )
    return rms


def compute_opening_score(depth_df, pre_window="2D", post_window="2D"):
    s = depth_df["raw_depth"].sort_index()

    # Pre-event max (lookback)
    pre_max = s.rolling(pre_window, min_periods=1, center=True).quantile(0.9)

    # Post-event median (lookforward)
    post_median = s[::-1].rolling(post_window, min_periods=1).quantile(0.9)[::-1]

    # Opening score
    score = pre_max - post_median
    return score


def extract_windowed_features(depth_df, window_size="1D", slope_window_size="2D", base_freq="6min"):
    """
    Extract features on sliding windows.

    window_size : e.g. '1D' or '3D'

    Each row is timestamped by the window end time.
    """
    df = depth_df.copy().sort_index()

    rms_tidal = bandpass_tidal_rms(depth_df, rolling_time=window_size)
    rms_tidal.name = "bandpass_tidal_rms"
    d_rms = detrended_rms(depth_df, window=window_size).rolling(window_size, center=True).mean()
    d_rms.name = "detrended_rms"
    d_slope = (
        depth_df["raw_depth"]
        .rolling(slope_window_size, center=True)
        .apply(water_slope)
        .rolling(window_size, center=True)
        .mean()
    )
    d_slope.name = "slope"
    var = depth_df["raw_depth"].rolling(window_size, center=True).var()
    var.name = "d_var"
    opening_score = compute_opening_score(depth_df)
    opening_score.name = "opening_score"

    features = (
        pd.concat(
            [
                rms_tidal,
                # d_rms,
                d_slope,
                var,
                opening_score,
            ],
            axis=1,
        )
        .resample("1h")
        .mean()
    )

    return features

In [ ]:
# Combine: only keep timestamps where both exist
daily_features = extract_windowed_features(
    depth_df,
    window_size="12h",
    slope_window_size="24h",
    base_freq="6min",
)

daily_features.head()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def minmax_norm(s):
    s = s.astype(float)
    s_min = s.min()
    s_max = s.max()
    if s_max == s_min:
        return pd.Series(0.5, index=s.index)  # flat if constant
    return (s - s_min) / (s_max - s_min)


def plot_features_with_depth(depth_df, feature_df, feature_cols, title=""):
    """
    Plot raw water depth and a set of normalized features on the same time axis.

    depth_df   : full-resolution water level (index = datetime, col 'raw_depth')
    feature_df : windowed features (index = timestamps of windows)
    feature_cols : list of column names from feature_df to plot
    """

    # Align depth to feature timestamps (e.g. daily/window end)
    depth_at_features = depth_df["raw_depth"].reindex(feature_df.index, method="nearest")

    # Normalize each feature to [0, 1]
    norm_features = feature_df[feature_cols].apply(minmax_norm)

    fig, ax_depth = plt.subplots(figsize=(16, 5))

    # --- Water depth (left axis) ---
    ax_depth.plot(
        depth_df.index, depth_df["raw_depth"], color="black", linewidth=1.2, label="Raw Depth"
    )
    ax_depth.set_ylabel("Raw Depth", color="black")
    ax_depth.tick_params(axis="y", labelcolor="black")

    # --- Normalized features (right axis) ---
    ax_feat = ax_depth.twinx()
    colors = plt.cm.tab10(np.linspace(0, 1, len(feature_cols)))

    for col, c in zip(feature_cols, colors):
        ax_feat.plot(
            norm_features.index,
            norm_features[col],
            label=col,
            color=c,
            linewidth=1.2,
        )

    ax_feat.set_ylabel("Normalized feature value (0–1)")
    ax_feat.set_ylim(-0.05, 1.05)

    # Title & legend
    ax_depth.set_title(title)
    lines1, labels1 = ax_depth.get_legend_handles_labels()
    lines2, labels2 = ax_feat.get_legend_handles_labels()
    ax_feat.legend(lines1 + lines2, labels1 + labels2, loc="upper right")

    plt.tight_layout()
    plt.show()

In [ ]:
from sklearn.preprocessing import StandardScaler

# Drop rows where features are missing
features_clean = daily_features.dropna().copy()

# Standardize feature matrix
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features_clean)

# Keep index for plotting later
feature_index = features_clean.index

In [ ]:
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture

# Number of clusters to try
n_clusters = 2

# --- KMeans ---
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init="auto")
kmeans_labels = kmeans.fit_predict(X_scaled)

# --- Gaussian Mixture Model ---
gmm = GaussianMixture(n_components=n_clusters, covariance_type="full", random_state=42)
gmm_labels = gmm.fit_predict(X_scaled)

# Add labels back into a DataFrame for easier plotting
cluster_results = cluster_daily = pd.DataFrame(
    {
        "acquired": feature_index,
        "kmeans_label": kmeans_labels,
        "gmm_label": gmm_labels,
    }
).set_index("acquired")

cluster_results.head(3)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# --- Ensure predictions have datetime and are sorted ---
pred = predictions.copy()
pred["acquired"] = pd.to_datetime(pred["acquired"])
pred = pred.sort_values("acquired")

# --- Align daily depth values to cluster_daily index (nearest sample) ---
depth_daily = depth_df["raw_depth"].reindex(cluster_daily.index, method="nearest")

fig, ax_depth = plt.subplots(figsize=(16, 6))

# 1. Full water depth time series (left axis)
ax_depth.plot(
    depth_df.index, depth_df["raw_depth"], color="black", linewidth=1, label="Raw Depth", alpha=0.1
)
ax_depth.set_ylabel("Raw Depth", color="black")
ax_depth.tick_params(axis="y", labelcolor="black")

# 2. Daily cluster states as colored markers (using GMM labels here)
label_col = "gmm_label"  # or "kmeans_label" if you prefer

unique_clusters = sorted(cluster_daily[label_col].unique())
colors = plt.cm.tab10(np.linspace(0, 1, len(unique_clusters)))

for cid, color in zip(unique_clusters, colors):
    mask = cluster_daily[label_col] == cid
    ax_depth.scatter(
        cluster_daily.index[mask],
        depth_daily[mask],  # daily depth aligned to cluster time
        s=10,
        color=color,
        alpha=0.9,
        edgecolor="none",
        label=f"Cluster {cid}",
    )

# 3. Second y-axis for prediction probabilities
ax_pred = ax_depth.twinx()
ax_pred.scatter(
    pred["acquired"],
    pred["y_prob"],
    color="C3",
    s=80,
    alpha=0.99,
    marker="+",
    label="Prediction Probability",
)
ax_pred.scatter(
    img_labels["acquired"],
    img_labels["label_idx"] + 0.05,
    color="C0",
    s=80,
    alpha=0.99,
    marker="^",
    label="Label",
)
ax_pred.set_ylabel("Predicted Probability", color="C3")
ax_pred.tick_params(axis="y", labelcolor="C3")

# Title and combined legend
plt.title("Water Level with Daily Cluster States and Model Predictions")

lines1, labels1 = ax_depth.get_legend_handles_labels()
lines2, labels2 = ax_pred.get_legend_handles_labels()
ax_depth.legend(lines1 + lines2, labels1 + labels2, loc="upper right")

plt.tight_layout()
plt.savefig("/Users/kyledorman/Desktop/water_depth_clustering_arroyo_de_la_laguna.png")
plt.show()

In [ ]:
import numpy as np
import pandas as pd

# Inverse transform cluster centers back into original feature units
km_centers_scaled = kmeans.cluster_centers_
km_centers = scaler.inverse_transform(km_centers_scaled)

cluster_centroids_km = pd.DataFrame(km_centers, columns=features_clean.columns)
cluster_centroids_km.index.name = "cluster"

cluster_centroids_km

In [ ]:
pred = predictions.copy()
pred["acquired"] = pd.to_datetime(pred["acquired"])
pred = pred.set_index("acquired").sort_index()

y_prob = pred[
    (pred.index >= cluster_daily.index.min()) & (pred.index <= cluster_daily.index.max())
][["y_prob", "source_tif"]]

# get nearest cluster label for each prediction timestamp
cluster_nearest = cluster_daily["kmeans_label"].reindex(y_prob.index, method="nearest")

plot_df = pd.DataFrame(
    {"y_prob": y_prob.y_prob, "source_tif": y_prob.source_tif, "cluster": cluster_nearest}
).dropna()

import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
sns.boxplot(data=plot_df, x="cluster", y="y_prob")
sns.stripplot(data=plot_df, x="cluster", y="y_prob", color="black", alpha=0.2)

plt.title("Distribution of Prediction Probabilities by Cluster")
plt.xlabel("Cluster ID")
plt.ylabel("Prediction Probability (y_prob)")

plt.show()

In [ ]:
row = plot_df[(plot_df.cluster == 0) & (plot_df.y_prob > 0.2)].iloc[4]

plt.figure()
with rasterio.open(row.source_tif) as src:
    data = src.read()
    nodata = src.read(1, masked=True).mask
    img = false_color(data, nodata)

plt.axis("off")
plt.imshow(img)
_ = plt.title(f"{row.y_prob:0.2f} {Path(row.source_tif).stem}")

In [ ]:
def normalize(x):
    xm = x.mean()
    xs = x.std()
    return (x - xm) / xs


def rolling_ls_slope_1h(series: pd.Series, window_hours: int) -> pd.Series:
    def _slope(x):
        t = np.arange(len(x))
        m, _ = np.polyfit(t, x, 1)
        return m  # units per hour (since spacing is 1h)

    return series.rolling(window_hours, center=True).apply(_slope, raw=True)


def bandpass_tidal_rms(
    s,
    low_cpd: float = 0.4,  # ~60 h period
    high_cpd: float = 1.2,  # ~20 h period
    rms_window: str = "12h",  # rolling RMS window
) -> pd.Series:
    """
    Band-pass filter around 'tidal-ish' frequencies and compute rolling RMS.

    low_cpd, high_cpd: cutoff freqs in cycles per day.
    rms_window: window (e.g. '6h', '12h') for RMS of the filtered signal.
    """
    # sampling interval in minutes
    dt_min = np.median(np.diff(s.index.values).astype("timedelta64[m]").astype(float))
    if not np.isfinite(dt_min) or dt_min <= 0:
        raise ValueError("Cannot infer a positive sampling interval from index")

    # samples per day
    fs_per_day = (24 * 60.0) / dt_min  # e.g. 240 samples/day for 6-min data
    nyq_cpd = 0.5 * fs_per_day  # Nyquist in cycles/day

    # normalize cutoff frequencies for butter (0..1 w.r.t. Nyquist)
    low = low_cpd / nyq_cpd
    high = high_cpd / nyq_cpd
    if not (0 < low < high < 1):
        raise ValueError(
            f"Normalized band [{low}, {high}] is invalid; "
            f"check low_cpd/high_cpd vs sampling (dt_min={dt_min})."
        )

    b, a = butter(N=4, Wn=[low, high], btype="band")
    filtered = filtfilt(b, a, s.values)

    # rolling RMS of bandpassed component
    rms = (
        pd.Series(filtered, index=s.index)
        .rolling(rms_window, center=True)
        .apply(lambda x: np.sqrt(np.mean(x**2)), raw=True)
    )
    rms.name = "tidal_band_rms"
    return rms


def calc_tidal_range(s):
    dmin = s.rolling("25h", center=True).quantile(0.01)
    dmax = s.rolling("25h", center=True).quantile(0.99)
    dvar = s.rolling("25h", center=True).var()
    d_diff = dmax - dmin
    d_tide = d_diff.rolling("2D", center=True).quantile(0.01)

    return d_tide


def compute_opening_score(
    s: pd.Series, pre_window: str = "2D", post_window: str = "2D"
) -> pd.Series:
    """
    For each timestamp t, compute:
        pre_median  = median over (t - pre_window, t)
        post_median = median over (t, t + post_window)
    and return pre_median - post_median.
    """
    s = s.sort_index()

    # Pre: use past values only (shift so current is excluded)
    pre_median = (
        s.shift(1)  # exclude current point
        .rolling(pre_window, min_periods=1)
        .median()
    )

    # Post: reverse, use "future" values only, then flip back
    post_median = (
        s[::-1]
        .shift(1)  # exclude current point in reversed time
        .rolling(post_window, min_periods=1)
        .median()[::-1]  # flip back to original order
    )

    score = pre_median - post_median
    score.name = "opening_score"
    return score


def build_hmm_features(depth_df, dt_minutes=6, depth_col="raw_depth", short_hours=3, long_hours=36):
    """
    depth_df: DataFrame with DatetimeIndex and 'raw_depth' column.
    Returns hourly features for HMM.
    """

    s = depth_df[depth_col].sort_index()

    # --- 1) Downsample depth to 1-hour (optionally with smoothing) ---
    # Smooth over 1h then take hourly mean
    depth_1h = s.rolling("1h", center=True).mean().resample("1h").mean().dropna()

    d_tide = calc_tidal_range(s.copy()).resample("1h").min()

    # --- 3) Opening / breach score (use your existing fn then downsample) ---
    opening_score = compute_opening_score(s)
    opening_score_1h = opening_score.resample("1h").mean().reindex(depth_1h.index)

    # --- 5) Level anomaly relative to long window mean ---
    d_mean = depth_1h.mean()
    rel_level = normalize(depth_1h - d_mean).rolling("48h").mean()

    daily_min = (
        s.rolling("24h", center=True)
        .quantile(0.01)
        .resample("1h")
        .quantile(0.01)
        .reindex(depth_1h.index)
        .rolling("24h", center=True)
        .mean()
    )

    daily_min_slope = (
        s.rolling("24h", center=True)
        .quantile(0.01)
        .rolling("48h", center=True)
        .apply(water_slope)
        .resample("1h")
        .quantile(0.99)
        .reindex(depth_1h.index)
        .rolling("12h", center=True)
        .mean()
    )
    m_daily_min = daily_min.mean()
    daily_min_rel = daily_min - m_daily_min

    breach_score = opening_score_1h.clip(lower=0.0)  # max(opening, 0)
    fill_score = (
        (-opening_score)
        .rolling("28h", center=True)
        .quantile(0.99)
        .resample("1h")
        .mean()
        .reindex(depth_1h.index)
    ).clip(lower=0.0)

    pos_slope = daily_min_slope.clip(lower=0.0)
    neg_slope = (-daily_min_slope).clip(lower=0.0)

    # --- 6) Assemble features, align, drop NaNs ---
    feats = pd.concat(
        [
            # tidal_rms_1h,
            breach_score.rename("breach_score"),
            fill_score.rename("fill_score"),
            daily_min_rel.rename("rel_level"),
            d_tide.rename("tide_range"),
            # daily_min.rename("daily_min"),
            # pos_slope.rename("pos_slope"),
            # neg_slope.rename("neg_slope"),
        ],
        axis=1,
    ).dropna()

    return feats

In [ ]:
import matplotlib.pyplot as plt


def plot_hmm_features(depth: pd.Series, features: pd.DataFrame):
    t_idx = features.index
    depth_aligned = depth.reindex(t_idx)
    N = len(features.columns)

    colors = cm.tab10(np.linspace(0, 1, N))

    fig, axes = plt.subplots(N, 1, figsize=(10, 3 * N), sharex=True)

    for r, c in enumerate(features.columns):
        ax = axes[r]
        ax.plot(depth.index, depth.values, lw=1, alpha=0.4, label="Water level (full)")
        ax.axhline(0, color="k", lw=0.5)
        ax.set_ylabel("Water Depth")
        ax.set_title(c)
        ax.set_ylim(depth.min(), depth.max())

        ax2 = ax.twinx()
        ax2.plot(features[c].index, features[c], color=colors[r], linewidth=1.2)
        ax2.set_ylabel(c)
        ax2.set_ylim(features[c].min(), features[c].max())

    fig.tight_layout()
    # fig.savefig("/Users/kyledorman/Desktop/hmm_features.png")
    plt.show()


depth_hmm = depth_df["raw_depth"].resample("1h").mean()
features = build_hmm_features(depth_df, short_hours=3, long_hours=24 * 7)
plot_hmm_features(depth_hmm, features)

In [ ]:
from hmmlearn.hmm import GaussianHMM
from sklearn.preprocessing import StandardScaler


def fit_hmm_from_features(features: pd.DataFrame, n_states: int = 4, random_state: int = 0):
    X = features.values
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    hmm = GaussianHMM(
        n_components=n_states,
        covariance_type="full",
        n_iter=300,
        random_state=random_state,
        verbose=False,
    )
    hmm.fit(X_scaled)

    state_seq = hmm.predict(X_scaled)
    posteriors = hmm.predict_proba(X_scaled)

    return {
        "model": hmm,
        "scaler": scaler,
        "state_seq": state_seq,
        "posteriors": posteriors,
    }

In [ ]:
depth_hmm = depth_df["raw_depth"].resample("1h").mean()
features = build_hmm_features(depth_df, short_hours=3, long_hours=24 * 7)
n_states = 4
result = fit_hmm_from_features(features, n_states=n_states)
hmm_model = result["model"]
scaler = result["scaler"]
post = result["posteriors"]
state_seq = result["state_seq"]
t_idx = features.index

In [ ]:
state_means_scaled = hmm_model.means_
state_means = scaler.inverse_transform(state_means_scaled)
cols = features.columns.tolist()

for k, m in enumerate(state_means):
    print(f"\nState {k}")
    for val, name in zip(m, cols):
        print(f"  {name:12s} = {val: .4f}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def plot_hmm_results(depth: pd.Series, features: pd.DataFrame, posteriors, state_seq, save_path):
    """
    depth: full-resolution depth series
    features: DataFrame of features used for HMM (aligned to 1H usually)
    posteriors: posterior probabilities (T x K)
    state_seq: Viterbi MAP state sequence (T)
    """
    t_idx = features.index
    depth_aligned = depth.reindex(t_idx)

    n_states = posteriors.shape[1]
    n_features = features.shape[1]
    state_colors = cm.tab10(np.linspace(0, 1, n_states))
    feat_colors = cm.tab10(np.linspace(0, 1, n_features))

    # Total rows:
    # 1 = water level + HMM states
    # n_features = each feature gets its own row
    # 1 = state probabilities
    total_rows = 1 + n_features + 1

    fig, axes = plt.subplots(total_rows, 1, figsize=(12, 3 * total_rows), sharex=True)

    row = 0

    # ----------------------------------------------------------
    # (1) Water depth with HMM regime colors
    # ----------------------------------------------------------
    ax0 = axes[row]
    row += 1

    ax0.plot(depth.index, depth.values, lw=1, alpha=0.4, label="Water level (full)")
    ax0.scatter(
        t_idx,
        depth_aligned.values,
        c=state_seq,
        s=5,
        cmap="tab10",
        alpha=0.9,
    )

    ax0.set_ylabel("Depth")
    ax0.set_title("Water level with HMM states")

    handles = [
        plt.Line2D([0], [0], marker="o", linestyle="", color=state_colors[k], label=f"State {k}")
        for k in range(n_states)
    ]
    ax0.legend(handles=handles, loc="upper left")

    # ----------------------------------------------------------
    # (2) Each feature in its own subplot
    # ----------------------------------------------------------
    for i, col in enumerate(features.columns):
        ax = axes[row]
        row += 1

        # Plot raw depth for context
        ax.plot(depth.index, depth.values, lw=1, alpha=0.25, color="gray")
        ax.set_ylabel("Depth")
        ax.set_ylim(depth.min(), depth.max())
        ax.axhline(0, color="k", lw=0.4)

        # Feature overlay on twin axis
        ax2 = ax.twinx()
        ax2.plot(features.index, features[col], color=feat_colors[i], lw=1.4)
        ax2.set_ylabel(col)
        ax2.set_ylim(features[col].min(), features[col].max())

        ax.set_title(f"Feature: {col}")

    # ----------------------------------------------------------
    # (3) Posterior state probabilities
    # ----------------------------------------------------------
    axp = axes[row]

    for k in range(n_states):
        axp.plot(t_idx, posteriors[:, k], lw=1.5, color=state_colors[k], label=f"State {k}")

    axp.set_ylim(-0.02, 1.02)
    axp.set_ylabel("P(state)")
    axp.set_xlabel("Time")
    axp.set_title("HMM state probabilities")
    axp.legend(loc="upper left", ncol=2)

    fig.tight_layout()
    if save_path is not None:
        fig.savefig(save_path)
    plt.show()


# usage
plot_hmm_results(depth_hmm, features, post, state_seq, save_path=None)

In [ ]:
b = features["breach_score"]
f = features["fill_score"]
tr = features["tide_range"]
rl = features["rel_level"]

# thresholds – you can tune these by eyeballing histograms
B = 0.15  # b.quantile(0.3)   # 'big' breach
F = 0.2  # f.quantile(0.2)   # 'big' fill
T = 0.2  # tr.quantile(0.2)  # 'strong tidal'
RL = 0.15
RL_low = -0.1

In [ ]:
labels = pd.Series("other", index=features.index)

# strong breach
labels[(b > B) & (b > f)] = "breach"

# strong fill
labels[(f > F) & (f > b)] = "fill"

# tidal-ish (oscillatory, not strongly filling or breaching)
labels[(tr > T) & (b < B) & (f < F)] = "tidal"

# flat-high vs flat-low (optional)
labels[(tr <= T) & (b < B) & (f < F) & (rl > RL)] = "flat_high"
# labels[(tr <= T) & (b < B) & (f < F) & (rl <= RL_low)] = "tidal"

In [ ]:
# if ambiguous (labels == 'other'), fall back to HMM state mapping
state_map = pd.Series(state_seq, index=features.index)

fallback_map = {
    0: "tidal",
    1: "flat_high",
    # 2: "tidal_or_mixed",
    # 3: "breach",
}

# labels = labels.copy()
# mask_other = labels == "other"
# labels[mask_other] = state_map[mask_other].map(fallback_map)

In [ ]:
import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np


def plot_label_bands(depth, labels, figsize=(14, 4), label_colors=None):
    depth = depth.sort_index()
    # depth = features["tide_range"].clip(0, 0.7)
    labels = labels.reindex(depth.index).ffill()

    uniq = labels.unique()
    if label_colors is None:
        cmap = plt.get_cmap("tab10")
        label_colors = {lab: cmap(i % 10) for i, lab in enumerate(uniq)}

    fig, ax = plt.subplots(figsize=figsize)

    # Plot water depth
    ax.plot(depth.index, depth.values, lw=1.2, color="black", alpha=0.7)
    ax.set_ylabel("Depth")
    ax.set_title("Water Level with Label Bands")

    # --- Draw segmented background bands ---
    start = depth.index[0]
    prev_label = labels.iloc[0]

    for t, lab in labels.iloc[1:].items():
        if lab != prev_label:
            ax.axvspan(start, t, color=label_colors[prev_label], alpha=0.25, zorder=-1)
            start = t
            prev_label = lab

    # final segment
    t_end = depth.index[-1]
    ax.axvspan(start, t_end, color=label_colors[prev_label], alpha=0.25, zorder=-1)

    # Legend
    handles = [patches.Patch(color=label_colors[lab], label=lab) for lab in uniq]
    ax.legend(handles=handles, loc="upper right")

    fig.tight_layout()
    plt.show()


plot_label_bands(depth_df["raw_depth"], labels)

In [ ]:
import numpy as np
import pandas as pd


def derive_open_closed_from_segments(
    depth: pd.Series,
    labels: pd.Series,
    high_flat_label: str = "flat_high",
    fill_label: str = "fill",
    breach_label: str = "breach",
    tidal_label: str = "tidal",
) -> pd.Series:
    """
    Convert string labels into open(1) / closed(0) / other(NaN) based on segment rules:
      - if current segment is high_flat: mark closed.
      - if current is fill and next segment is high_flat: mark closed, else open.
      - if current is breach: find max in segment, mark before as closed and after as open.
      - if current is tidal: mark open.
      - everything else stays NaN.
    """
    # Align & sort
    depth = depth.sort_index()
    labels = labels.sort_index().reindex(depth.index)

    n = len(labels)
    open_closed = pd.Series(np.nan, index=depth.index, dtype=float)

    if n == 0:
        return open_closed

    # Identify contiguous label segments
    label_vals = labels.values
    seg_starts = [0]
    for i in range(1, n):
        if label_vals[i] != label_vals[i - 1]:
            seg_starts.append(i)
    seg_starts = np.array(seg_starts, dtype=int)

    seg_ends = np.r_[seg_starts[1:] - 1, n - 1]  # inclusive ends

    n_segs = len(seg_starts)

    for seg_idx in range(n_segs):
        start = seg_starts[seg_idx]
        end = seg_ends[seg_idx]
        lab = label_vals[start]

        # Look at next segment's label (if any)
        next_lab = label_vals[seg_starts[seg_idx + 1]] if seg_idx + 1 < n_segs else None

        seg_slice = slice(start, end + 1)

        # Rule: high_flat -> closed
        if lab == high_flat_label:
            open_closed.iloc[seg_slice] = 0.0

        # Rule: fill -> closed if next is high_flat, else open
        elif lab == fill_label:
            if next_lab == high_flat_label or next_lab == breach_label:
                open_closed.iloc[seg_slice] = 0.0
            else:
                open_closed.iloc[seg_slice] = 1.0

        # Rule: breach -> before max closed, after max open
        elif lab == breach_label:
            seg_depth = depth.iloc[seg_slice]
            if not seg_depth.empty:
                # position of max within this segment
                local_max_pos = int(np.argmax(seg_depth.values)) + 1
                max_idx = start + local_max_pos

                # before max: closed
                open_closed.iloc[start:max_idx] = 0.0

                # after max : open
                open_closed.iloc[max_idx : end + 1] = 1.0

        # Rule: tidal -> open
        elif lab == tidal_label:
            open_closed.iloc[seg_slice] = 1.0

        # else: leave as NaN ("other")

    return open_closed


open_closed = derive_open_closed_from_segments(depth_hmm, labels)

# # Quick sanity plot with your existing helper
# plot_label_bands_with_states(
#     depth=depth_df["raw_depth"],
#     labels=labels,
#     state_seq=state_seq,  # if you want to overlay states
# )

# And then something like:
plot_label_bands(depth_df["raw_depth"], open_closed.map({0.0: "closed", 1.0: "open"}))

In [ ]:
pred = predictions.set_index("acquired").sort_index()
y_pred = pred["y_pred"]  # 0/1 predictions
gt = open_closed.dropna().sort_index()  # ground truth 0/1 labels
gt = gt.rename("open_closed")

df_pred = y_pred.reset_index()
df_gt = gt.reset_index()

matched = pd.merge_asof(
    df_pred,
    df_gt,
    on="acquired",
    direction="nearest",  # pick the closest label in time
    tolerance=pd.Timedelta("1h"),  # specify max allowed distance (optional)
)

matched = matched.dropna(subset=["open_closed"])
matched["correct"] = (matched["y_pred"] == matched["open_closed"]).astype(int)

accuracy = matched["correct"].mean()
accuracy

In [ ]:
# STEP 1 — Forward-fill predictions onto the depth_hmm index

pred_ser = predictions.set_index("acquired").sort_index()["y_pred"]

# Forward fill to the hourly index of depth_hmm
pred_filled = pred_ser.reindex(depth_hmm.index, method="ffill")

# Optional: remove values before first prediction
pred_filled[depth_hmm.index < pred_ser.index.min()] = pd.NA


# STEP 2 — Convert numeric predictions → labels for plotting

pred_labels = pred_filled.map({0: "closed", 1: "open"}).rename("pred_label")


# STEP 3 — Use your existing plot_label_bands function

plot_label_bands(depth_hmm, pred_labels)
plot_label_bands(depth_df["raw_depth"], open_closed.map({0.0: "closed", 1.0: "open"}))